# ✏️🗑️ HTTP PUT & DELETE — Completing the CRUD API (Final `main.py`)

**Goal:** learn how to **update** and **delete** records with FastAPI, and see the **final, complete Patient Management API**.

> Series: [Pydantic notes](Pydantic_Notes.ipynb) → [HTTP GET](HTTP_get_Concepts.ipynb) → [HTTP POST](HTTP_post_concepts.ipynb) → **PUT & DELETE (this notebook)**

## 📑 Table of Contents
1. Recap: CRUD and the final API map
2. Theory: PUT, PATCH and DELETE
3. The final `main.py` (heavily commented)
4. Test setup
5. `PatientUpdate` model and `exclude_unset`, the key trick
6. Testing `PUT /update/{patient_id}`
7. Testing `DELETE /delete/{patient_id}`
8. Bugs found in the live-class `main.py` (and fixes)
9. Running the server, Swagger UI, `curl`
10. Summary, common mistakes, practice exercises

---
# 1️⃣ Recap: CRUD and the final API map

**CRUD** = the four basic operations on data. Each maps to an HTTP method:

| CRUD | HTTP method | Endpoint in our API | Success code | Data goes in |
|------|-------------|---------------------|--------------|--------------|
| **C**reate | `POST` | `/create` | `201 Created` | JSON body (full record) |
| **R**ead | `GET` | `/view`, `/patient/{id}`, `/sort` | `200 OK` | URL (path / query) |
| **U**pdate | `PUT` | `/update/{patient_id}` | `200 OK` | URL (id) + JSON body (changed fields) |
| **D**elete | `DELETE` | `/delete/{patient_id}` | `200 OK` | URL (id) |

With PUT and DELETE our API is a **complete CRUD application**.

---
# 2️⃣ Theory: PUT, PATCH and DELETE

## PUT: update an existing resource
- The id of the resource is in the **URL** (`/update/P001`): *which* record to change.
- The **request body** carries the *new data*.
- **Idempotent:** sending the same PUT once or ten times leaves the server in the **same final state**.
- Returns `404 Not Found` if the resource does not exist, `422` if the new data is invalid.

### PUT vs PATCH (interview favourite)
| | **PUT** (by the book) | **PATCH** (by the book) |
|---|-----|-------|
| Meaning | **Replace** the whole resource | Change **part** of the resource |
| Body | must contain **all** fields | only the fields that change |
| Idempotent | ✅ always | usually, but not guaranteed |

> 📝 **Class note:** our `/update` endpoint changes only the fields the client sends. By the book that behaviour is **PATCH**; many real APIs still use PUT for it.
> It works the same. To follow the standard strictly you would use `@app.patch(...)`. (Practice exercise 1.)

## DELETE: remove a resource
- The id is in the URL, and there is normally **no body**.
- **Idempotent:** after the first delete the record is gone; deleting again does not change the state further (the 2nd call simply answers `404`).
- Success: `200 OK` with a message, or `204 No Content` (no body at all).
- ⚠️ **Destructive**: there is no undo. Real systems protect it with authentication and often use a *soft delete* (mark as deleted instead of removing).

## The update algorithm (what our PUT endpoint does)
```
client:  PUT /update/P001   {"weight": 70}
  1. load data
  2. id not found?                     → 404
  3. keep only the fields sent          → {"weight": 70}
  4. nothing valid to update?           → 400
  5. merge into the stored record       → old record + new weight
  6. validate the FULL record (Pydantic) → recalculates bmi & verdict   (invalid → error)
  7. save the whole dictionary
  8. return the updated patient         → 200
```

### Why a separate `PatientUpdate` model?
| | `Patient` | `PatientUpdate` |
|---|-----------|-----------------|
| Used by | POST /create | PUT /update |
| Fields required? | **all required** | **all optional** (`= None`) |
| Has `id`? | yes | no (id is in the URL) |
| Has `bmi`/`verdict`? | computed | no (recalculated after merge) |

If we reused `Patient` for updates, the client would be forced to re-send every field just to fix a city name.

### `exclude_unset` vs `exclude_none` (from the Pydantic notes)
| Option | Removes fields that... | Why we need it here |
|--------|------------------------|---------------------|
| `exclude_unset=True` | were **not sent** by the client | so we don't overwrite stored data with `None` defaults |
| `exclude_none=True` | are `None` (including an **explicit** `null`) | so `{"age": null}` cannot wipe out a valid age |

---
# 3️⃣ The final `main.py`

The cell below writes the complete API to `main.py`. Every endpoint from all lessons is here.
It is the live-class code plus a few fixes, all listed in **section 8**.

In [1]:
%%writefile main.py
"""
Patient Management System API  (FastAPI + Pydantic)

Endpoints
    GET    /                        health check
    GET    /about                   about message
    GET    /view                    all patients
    GET    /patient/{patient_id}    one patient
    GET    /sort                    sorted patients (?sort_by=&order=)
    POST   /create                  create a patient          (Create)
    PUT    /update/{patient_id}     update given fields       (Update)
    DELETE /delete/{patient_id}     delete a patient          (Delete)

Run:  uvicorn main:app --reload      then open  http://127.0.0.1:8000/docs
"""
import json
from typing import Annotated, Literal, Optional

from fastapi import FastAPI, HTTPException, Path, Query
from pydantic import BaseModel, Field, ValidationError, computed_field, field_validator

app = FastAPI(
    title="Patient Management System API",
    description="A REST API to manage patient records using FastAPI and Pydantic.",
    version="1.0.0",
)

# The JSON file that acts as our tiny database.
DATA_FILE = "patients.json"


# ───────────────────────────── Pydantic models ─────────────────────────────

class Patient(BaseModel):
    """A COMPLETE patient record (used for create + for reading/validating stored data)."""

    id: Annotated[str, Field(..., description="ID of the patient", examples=["P001"])]
    name: Annotated[str, Field(..., description="Name of the patient")]
    city: Annotated[str, Field(..., description="City where the patient is living")]
    age: Annotated[int, Field(..., gt=0, lt=120, description="Age of the patient")]
    gender: Annotated[Literal["male", "female", "others"], Field(..., description="Gender of the patient")]
    height: Annotated[float, Field(..., gt=0, description="Height of the patient in meters")]
    weight: Annotated[float, Field(..., gt=0, description="Weight of the patient in kilograms")]

    # Clean the RAW gender text BEFORE the Literal check runs, so 'Female', 'MALE', ' other ' all work.
    # This also protects us from old records already saved with a capital letter (e.g. 'Female').
    @field_validator("gender", mode="before")
    @classmethod
    def normalize_gender(cls, value):
        if isinstance(value, str):
            value = value.strip().lower()
            if value == "other":          # accept the singular spelling too
                value = "others"
        return value

    # Calculated by the server from height & weight; the client never sends these.
    @computed_field
    @property
    def bmi(self) -> float:
        return round(self.weight / (self.height ** 2), 2)

    @computed_field
    @property
    def verdict(self) -> str:
        if self.bmi < 18.5:
            return "Underweight"
        elif self.bmi < 25:
            return "Normal"
        elif self.bmi < 30:
            return "Overweight"
        return "Obese"


class PatientUpdate(BaseModel):
    """
    Body of a PUT request: every field is OPTIONAL, so the client sends only what changes.
    There is no `id` (the id comes from the URL) and no bmi/verdict (calculated by the server).
    The rules (gt, lt, Literal) are repeated so a provided value is still validated.
    """

    name: Optional[str] = Field(default=None, description="Name of the patient")
    city: Optional[str] = Field(default=None, description="City where the patient is living")
    age: Optional[int] = Field(default=None, gt=0, lt=120, description="Age of the patient")
    gender: Optional[Literal["male", "female", "others"]] = Field(default=None, description="Gender of the patient")
    height: Optional[float] = Field(default=None, gt=0, description="Height in meters")
    weight: Optional[float] = Field(default=None, gt=0, description="Weight in kilograms")


# ───────────────────────────── JSON file helpers ─────────────────────────────

def load_data() -> dict:
    """Read all patients ({id: record}); return an empty dict if the file does not exist yet."""
    try:
        with open(DATA_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        return {}


def save_data(data: dict) -> None:
    """Write the COMPLETE dictionary back (mode 'w' overwrites the whole file)."""
    with open(DATA_FILE, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)


# ───────────────────────────── GET endpoints ─────────────────────────────

@app.get("/")
def hello():
    return {"message": "Patient Management System API"}


@app.get("/about")
def about():
    return {"message": "A fully functional API to manage patient records"}


@app.get("/view")
def view():
    """Return every patient exactly as stored."""
    return load_data()


@app.get("/patient/{patient_id}")
def view_patient(
    patient_id: Annotated[str, Path(description="ID of the patient in the database", examples=["P001"])]
):
    """Return one patient, or 404 if the id does not exist."""
    data = load_data()

    if patient_id not in data:
        raise HTTPException(status_code=404, detail="Patient not found")

    # Rebuild the model so bmi / verdict are always recalculated from height & weight.
    return Patient(id=patient_id, **data[patient_id])


@app.get("/sort")
def sort_patients(
    sort_by: Annotated[str, Query(description="Sort by height, weight, or bmi", examples=["bmi"])],
    # NOTE: with Annotated, the default value goes AFTER the "=", never inside Query(...).
    order: Annotated[str, Query(description="Sort order: asc or desc")] = "asc",
):
    """Return all patients sorted by height, weight or bmi."""
    valid_fields = ["height", "weight", "bmi"]

    if sort_by not in valid_fields:
        raise HTTPException(status_code=400, detail=f"Invalid field. Choose from {valid_fields}")
    if order not in ["asc", "desc"]:
        raise HTTPException(status_code=400, detail="Invalid order. Choose 'asc' or 'desc'")

    data = load_data()
    patients = [Patient(id=pid, **info) for pid, info in data.items()]

    # getattr(patient, 'bmi') → works for normal AND computed fields
    return sorted(patients, key=lambda p: getattr(p, sort_by), reverse=(order == "desc"))


# ───────────────────────────── POST: create ─────────────────────────────

@app.post("/create", status_code=201)          # 201 Created is the default success code for this endpoint
def create_patient(patient: Patient):
    """The JSON body is validated against `Patient` before this function runs (else 422)."""
    data = load_data()

    if patient.id in data:
        raise HTTPException(status_code=409, detail="Patient already exists")

    # id is the dictionary key, so it is left out of the stored record
    data[patient.id] = patient.model_dump(exclude={"id"})
    save_data(data)

    return {"message": "Patient created successfully", "patient": patient}


# ───────────────────────────── PUT: update ─────────────────────────────

@app.put("/update/{patient_id}")
def update_patient(patient_id: str, changes: PatientUpdate):
    """Update ONLY the fields that the client sent; keep everything else unchanged."""
    data = load_data()

    # 1) The patient must exist
    if patient_id not in data:
        raise HTTPException(status_code=404, detail="Patient not found")

    # 2) Only the fields the client really sent.
    #    exclude_unset=True → drop fields that were not in the request body
    #    exclude_none=True  → also drop explicit nulls (a null would otherwise overwrite good data
    #                         with None and crash the validation below with a 500 error)
    changes_dict = changes.model_dump(exclude_unset=True, exclude_none=True)

    if not changes_dict:
        raise HTTPException(status_code=400, detail="No valid fields provided to update")

    # 3) Merge: start from the stored record and overwrite the changed fields
    merged = data[patient_id].copy()
    merged.update(changes_dict)

    # 4) Validate the COMPLETE updated record (also recalculates bmi & verdict).
    #    We build this model by hand, so a bad stored value would raise a plain Python error (HTTP 500).
    #    Catch it and answer with a readable 422 instead.
    try:
        updated_patient = Patient(id=patient_id, **merged)
    except ValidationError as e:
        raise HTTPException(
            status_code=422,
            detail=e.errors(include_url=False, include_context=False, include_input=False),
        )

    # 5) Save (id stays the key, so exclude it from the stored record)
    data[patient_id] = updated_patient.model_dump(exclude={"id"})
    save_data(data)

    return {"message": "Patient updated successfully", "patient": updated_patient}


# ───────────────────────────── DELETE ─────────────────────────────

@app.delete("/delete/{patient_id}")
def delete_patient(patient_id: str):
    """Remove a patient permanently."""
    data = load_data()

    if patient_id not in data:
        raise HTTPException(status_code=404, detail="Patient not found")

    del data[patient_id]      # remove the key (and its record) from the dict
    save_data(data)           # persist the change

    return {"message": "Patient deleted successfully"}

Writing main.py


---
# 4️⃣ Test setup

We import the app from the file we just wrote and call it with `TestClient` (no server needed).
`main.DATA_FILE` is pointed to a **demo file**, so your real `patients.json` is never touched.

In [2]:
# %pip install fastapi uvicorn httpx2 "pydantic[email]"     # uncomment if needed (older versions: httpx)

import json
from fastapi.testclient import TestClient

import main                                  # the main.py written above
from main import Patient, PatientUpdate

main.DATA_FILE = 'patients_demo.json'        # use a demo file for this notebook
client = TestClient(main.app)


def show(response):
    """Helper: print status code and JSON body neatly."""
    print("Status code:", response.status_code)
    print(json.dumps(response.json(), indent=2))


# Start every run from the same 3 patients (this cell RESETS patients_demo.json)
seed = [
    Patient(id='P001', name='Ananya Verma', city='Delhi',     age=28, gender='female', height=1.65, weight=90),
    Patient(id='P002', name='Rohan Mehta',  city='Mumbai',    age=35, gender='male',   height=1.80, weight=72),
    Patient(id='P003', name='Sita Sharma',  city='Kathmandu', age=42, gender='female', height=1.58, weight=48),
]
main.save_data({p.id: p.model_dump(exclude={'id'}) for p in seed})

print("Patients in DB:", list(main.load_data().keys()))

Patients in DB: ['P001', 'P002', 'P003']


---
# 5️⃣ `PatientUpdate` and `exclude_unset`, the key trick

Every field of `PatientUpdate` defaults to `None`. Look at what `model_dump()` gives when the client sends **only** `weight`:

In [3]:
changes = PatientUpdate(weight=70)          # what FastAPI builds from the body {"weight": 70}

print("1) plain dump          :", changes.model_dump())
print("2) exclude_unset=True  :", changes.model_dump(exclude_unset=True))

1) plain dump          : {'name': None, 'city': None, 'age': None, 'gender': None, 'height': None, 'weight': 70.0}
2) exclude_unset=True  : {'weight': 70.0}


- (1) contains `None` for everything the client did **not** send. Merging this would **erase** the stored name, city, age, ...
- (2) contains **only** what was really sent → safe to merge.

Now the **merge**, done with `dict.update()` (it overwrites matching keys and keeps the others):

In [4]:
stored = main.load_data()['P001'].copy()
print("stored before :", {k: stored[k] for k in ('city', 'weight', 'bmi', 'verdict')})

stored.update(changes.model_dump(exclude_unset=True))          # only 'weight' changes
print("after merge   :", {k: stored[k] for k in ('city', 'weight', 'bmi', 'verdict')})
print("→ bmi/verdict are still OLD here; rebuilding the Patient model recalculates them:")

rebuilt = Patient(id='P001', **stored)         # validation + computed fields run again
print("after rebuild :", {'weight': rebuilt.weight, 'bmi': rebuilt.bmi, 'verdict': rebuilt.verdict})

stored before : {'city': 'Delhi', 'weight': 90.0, 'bmi': 33.06, 'verdict': 'Obese'}
after merge   : {'city': 'Delhi', 'weight': 70.0, 'bmi': 33.06, 'verdict': 'Obese'}
→ bmi/verdict are still OLD here; rebuilding the Patient model recalculates them:
after rebuild : {'weight': 70.0, 'bmi': 25.71, 'verdict': 'Overweight'}


That is the heart of the endpoint: **merge → rebuild the full `Patient` → save**. Rebuilding both *re-validates* the complete record and *refreshes* the computed fields.

---
# 6️⃣ Testing `PUT /update/{patient_id}`

### ✅ Scenario 1: update one field (weight 90 → 70)
Ananya's BMI and verdict must change automatically.

In [5]:
print("BEFORE:", {k: main.load_data()['P001'][k] for k in ('weight', 'bmi', 'verdict')})

show(client.put('/update/P001', json={'weight': 70}))

BEFORE: {'weight': 90.0, 'bmi': 33.06, 'verdict': 'Obese'}
Status code: 200
{
  "message": "Patient updated successfully",
  "patient": {
    "id": "P001",
    "name": "Ananya Verma",
    "city": "Delhi",
    "age": 28,
    "gender": "female",
    "height": 1.65,
    "weight": 70.0,
    "bmi": 25.71,
    "verdict": "Overweight"
  }
}


### ✅ Scenario 2: update several fields; the untouched fields stay as they were

In [6]:
show(client.put('/update/P002', json={'city': 'Pune', 'age': 36}))

Status code: 200
{
  "message": "Patient updated successfully",
  "patient": {
    "id": "P002",
    "name": "Rohan Mehta",
    "city": "Pune",
    "age": 36,
    "gender": "male",
    "height": 1.8,
    "weight": 72.0,
    "bmi": 22.22,
    "verdict": "Normal"
  }
}


### ✅ Scenario 3: PUT is **idempotent**
Sending the same request again gives the same final state.

In [7]:
client.put('/update/P002', json={'city': 'Pune', 'age': 36})
state_1 = main.load_data()

client.put('/update/P002', json={'city': 'Pune', 'age': 36})     # exactly the same request again
state_2 = main.load_data()

print("Same state after repeating the PUT:", state_1 == state_2)

Same state after repeating the PUT: True


### ❌ Scenario 4: unknown patient → `404`

In [8]:
show(client.put('/update/P999', json={'age': 30}))

Status code: 404
{
  "detail": "Patient not found"
}


### ❌ Scenario 5: invalid values → `422` (Pydantic validates the body before our code runs)

In [9]:
show(client.put('/update/P001', json={'age': 500}))                 # violates lt=120

Status code: 422
{
  "detail": [
    {
      "type": "less_than",
      "loc": [
        "body",
        "age"
      ],
      "msg": "Input should be less than 120",
      "input": 500,
      "ctx": {
        "lt": 120
      }
    }
  ]
}


In [10]:
show(client.put('/update/P001', json={'gender': 'robot'}))          # not in the Literal

Status code: 422
{
  "detail": [
    {
      "type": "literal_error",
      "loc": [
        "body",
        "gender"
      ],
      "msg": "Input should be 'male', 'female' or 'others'",
      "input": "robot",
      "ctx": {
        "expected": "'male', 'female' or 'others'"
      }
    }
  ]
}


### ⚠️ Scenario 6: empty body or explicit `null` → `400`
Nothing valid to update. Without `exclude_none=True` the `null` would overwrite the age with `None` (see section 8).

In [11]:
show(client.put('/update/P001', json={}))

Status code: 400
{
  "detail": "No valid fields provided to update"
}


In [12]:
show(client.put('/update/P001', json={'age': None}))

Status code: 400
{
  "detail": "No valid fields provided to update"
}


### 🔒 Scenario 7: the client cannot change the id, bmi or verdict
`PatientUpdate` has no `id`, `bmi` or `verdict` fields, so extra keys in the body are **ignored**.

In [13]:
before = main.load_data()['P003']

show(client.put('/update/P003', json={'city': 'Pokhara', 'id': 'HACKED', 'bmi': 999}))

print("ids in DB:", list(main.load_data().keys()))        # still P001, P002, P003
print("P003 bmi :", main.load_data()['P003']['bmi'], "(was", before['bmi'], ", unchanged; not 999)")

Status code: 200
{
  "message": "Patient updated successfully",
  "patient": {
    "id": "P003",
    "name": "Sita Sharma",
    "city": "Pokhara",
    "age": 42,
    "gender": "female",
    "height": 1.58,
    "weight": 48.0,
    "bmi": 19.23,
    "verdict": "Normal"
  }
}
ids in DB: ['P001', 'P002', 'P003']
P003 bmi : 19.23 (was 19.23 , unchanged; not 999)


---
# 7️⃣ Testing `DELETE /delete/{patient_id}`

```python
del data[patient_id]     # remove the key (and its record) from the dictionary
save_data(data)          # persist: without this the deletion would be lost
```

### ✅ Scenario 1: delete an existing patient

In [14]:
print("Before:", list(main.load_data().keys()))
show(client.delete('/delete/P003'))
print("After :", list(main.load_data().keys()))

Before: ['P001', 'P002', 'P003']
Status code: 200
{
  "message": "Patient deleted successfully"
}
After : ['P001', 'P002']


### Confirm with GET: the patient is really gone (`404`)

In [15]:
show(client.get('/patient/P003'))

Status code: 404
{
  "detail": "Patient not found"
}


### ❌ Scenario 2: delete it again → `404`
The **state** of the server is the same as after the first delete, which is why DELETE counts as *idempotent* (only the response differs).

In [16]:
show(client.delete('/delete/P003'))

Status code: 404
{
  "detail": "Patient not found"
}


### 🔁 Full CRUD lifecycle in one go
Create → Read → Update → Delete for a brand-new patient:

In [17]:
new = {'id': 'P010', 'name': 'Kabir Singh', 'city': 'Pune', 'age': 40, 'gender': 'male', 'height': 1.75, 'weight': 70}

print("C  POST  /create      →", client.post('/create', json=new).status_code)
print("R  GET   /patient/P010 →", client.get('/patient/P010').json()['verdict'])
print("U  PUT   /update/P010  →", client.put('/update/P010', json={'weight': 95}).json()['patient']['verdict'])
print("D  DELETE /delete/P010 →", client.delete('/delete/P010').status_code)
print("R  GET   /patient/P010 →", client.get('/patient/P010').status_code, "(gone)")

C  POST  /create      → 201
R  GET   /patient/P010 → Normal
U  PUT   /update/P010  → Obese
D  DELETE /delete/P010 → 200
R  GET   /patient/P010 → 404 (gone)


### 🔎 Bonus: `/sort` and `/view` on the final data
(`/sort` now returns full `Patient` objects with the computed `bmi`, sorted with `getattr`.)

In [18]:
for p in client.get('/sort', params={'sort_by': 'bmi', 'order': 'desc'}).json():
    print(f"{p['id']}  {p['name']:<14} bmi={p['bmi']:<6} {p['verdict']}")

P001  Ananya Verma   bmi=25.71  Overweight
P002  Rohan Mehta    bmi=22.22  Normal


---
# 8️⃣ Bugs found in the live-class `main.py` (and fixes)

I ran the code you wrote in class against every endpoint. Two were **real bugs** (a third one, #3, appeared later in your own server log; see below):

| # | Severity | Live-class code | Problem | Fix |
|---|----------|-----------------|---------|-----|
| 1 | 🔴 **App crashes on start** | `order: Annotated[str, Query("asc", description=...)] = "asc"` | FastAPI does not allow a default **inside** `Query()` when `Annotated` is used *and* another default after `=`: `AssertionError: Query default value cannot be set in Annotated`. The whole app fails to import. | `Annotated[str, Query(description=...)] = "asc"` |
| 2 | 🔴 **500 Internal Server Error** | `changes.model_dump(exclude_unset=True)` | `{"age": null}` is *set* (so it is kept), overwrites the stored age with `None`, then `Patient(...)` is built by hand inside the function and raises an **uncaught** `ValidationError` → 500. | `model_dump(exclude_unset=True, exclude_none=True)` + `400` if nothing is left |

Smaller improvements:

| # | Change | Reason |
|---|--------|--------|
| 3 | parameter `update_patient: Update_Patient` (same name as the function!) → `changes: PatientUpdate` | A parameter that shadows its own function name is confusing. Class names should be `CamelCase` (PEP 8). |
| 4 | `DATA_FILE = "patients.json"` constant | one place to change the file name (used by this notebook for the demo file) |
| 5 | `Path(..., description=...)` → `Path(description=...)` | with `Annotated`, `...` is unnecessary (path parameters are always required) |
| 6 | docstrings + type hints on helpers | documentation shows up in Swagger and in `help()` |
| 7 | `gender` normaliser + `ValidationError` → 422 in PUT | fixes bug #3 (case-sensitive `Literal` vs stored `'Female'`) |

Let's **prove** the two bugs.

### Proof of bug #1: the crash

In [19]:
from typing import Annotated
from fastapi import FastAPI, Query

scratch_app = FastAPI()

try:
    @scratch_app.get('/demo')
    def demo(order: Annotated[str, Query("asc", description="Sort order")] = "asc"):   # ← the live-class pattern
        return order
except AssertionError as e:
    print("AssertionError:", e)

AssertionError: `Query` default value cannot be set in `Annotated` for 'order'. Set the default value with `=` instead.


### Proof of bug #2: what a `null` does to the merge

In [20]:
from pydantic import ValidationError

stored = main.load_data()['P001'].copy()

# Old behaviour: the explicit null is "set", so exclude_unset keeps it
old_changes = PatientUpdate(age=None).model_dump(exclude_unset=True)
print("old dump               :", old_changes)

stored.update(old_changes)                               # age becomes None
try:
    Patient(id='P001', **stored)                         # built inside the endpoint by hand
except ValidationError as e:
    print("→ uncaught ValidationError inside the endpoint => HTTP 500")
    print(e.errors()[0]['loc'], e.errors()[0]['msg'])

# New behaviour
new_changes = PatientUpdate(age=None).model_dump(exclude_unset=True, exclude_none=True)
print("\nnew dump               :", new_changes, "→ empty → endpoint answers 400")

old dump               : {'age': None}
→ uncaught ValidationError inside the endpoint => HTTP 500
('age',) Input should be a valid integer

new dump               : {} → empty → endpoint answers 400


> 💡 **Lesson:** Pydantic models that FastAPI builds from the *request* are validated automatically (→ 422). But a model you build **yourself** inside the function (`Patient(id=..., **merged)`) raises a normal Python exception if the data is bad, so make sure bad data can't reach it.

### 🐞 Bug #3: a real 500 from your own server log
While running the server you got:
```
PUT /update/P005  →  500 Internal Server Error
pydantic_core.ValidationError: 1 validation error for Patient
gender
  Input should be 'male', 'female' or 'others' [input_value='Female', input_type=str]
```
**Cause:** the *stored* record of P005 has `"gender": "Female"` (capital F). Our `Literal["male","female","others"]` is **case-sensitive**, so when the endpoint rebuilds `Patient(...)` from the stored data, validation fails. The request itself was fine; the **old data** was the problem.
Every endpoint that rebuilds `Patient` from the file (`/patient/{id}`, `/sort`, `PUT`) fails the same way for such records.

**Fix (two layers):**
1. `@field_validator("gender", mode="before")` normalises the raw text (`strip().lower()`, and `other` → `others`) **before** the `Literal` check, so `Female` becomes `female`. Old records now load fine, and clients can send `Male`/`FEMALE` too.
2. The PUT endpoint catches `ValidationError` and answers with a readable **422** instead of a 500.

Let's reproduce it and confirm the fix:

In [21]:
from typing import Literal
from pydantic import BaseModel, ValidationError

# The OLD model (no normaliser): case-sensitive Literal
class OldPatient(BaseModel):
    gender: Literal['male', 'female', 'others']

try:
    OldPatient(gender='Female')                       # what the stored P005 record contained
except ValidationError as e:
    print("OLD model  →", e.errors()[0]['msg'])

OLD model  →

 Input should be 'male', 'female' or 'others'


In [22]:
# Simulate an old record saved with a capital letter
data = main.load_data()
data['P001']['gender'] = 'Female'                     # bad casing, like the P005 record in your file
main.save_data(data)

# The fixed API now reads it correctly...
print("GET  /patient/P001 →", client.get('/patient/P001').json()['gender'])

# ...and the PUT that used to give 500 now works, and saves the cleaned value
r = client.put('/update/P001', json={'city': 'Jaipur'})
print("PUT  /update/P001  →", r.status_code, "| stored gender:", main.load_data()['P001']['gender'])

GET  /patient/P001 → female
PUT  /update/P001  → 200 | stored gender: female


### 🧹 One-time clean-up of an existing `patients.json`
The validator fixes records **when they are updated**. To clean the whole file once, run this in your project folder (it only touches the `gender` text):
```python
import json

with open("patients.json", "r", encoding="utf-8") as f:
    data = json.load(f)

for record in data.values():
    g = record["gender"].strip().lower()
    record["gender"] = "others" if g == "other" else g

with open("patients.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=4)
```
> Make a copy of `patients.json` before running it.

---
# 9️⃣ Running the server, Swagger UI, `curl`

```bash
uvicorn main:app --reload
```
Open **`http://127.0.0.1:8000/docs`**: you will see all 8 endpoints. Try **PUT /update/{patient_id}** → *Try it out* → enter `P001` and a body such as `{"weight": 70}` → *Execute*.

### `requests`
```python
import requests
BASE = "http://127.0.0.1:8000"

requests.put(f"{BASE}/update/P001", json={"weight": 70}).json()
requests.delete(f"{BASE}/delete/P001").json()
```

### `curl`
```bash
curl -X PUT    http://127.0.0.1:8000/update/P001 -H "Content-Type: application/json" -d '{"weight": 70}'
curl -X DELETE http://127.0.0.1:8000/delete/P001
```
> The real server uses `patients.json` in the folder where you start `uvicorn` (start it from the folder that contains the file).

---
# 🔟 Summary

## 🧠 Cheat-sheet
| Concept | Code | Notes |
|---------|------|-------|
| PUT endpoint | `@app.put('/update/{patient_id}')` | id in URL, changes in body |
| Two parameters | `patient_id: str, changes: PatientUpdate` | path param + JSON body together |
| Optional-fields model | `Optional[T] = Field(default=None, ...)` | client sends only what changes |
| Only-sent fields | `model_dump(exclude_unset=True, exclude_none=True)` | never overwrite with `None` |
| Merge | `merged = old.copy(); merged.update(changes)` | keeps untouched fields |
| Re-validate + recompute | `Patient(id=patient_id, **merged)` | full validation, new bmi/verdict |
| DELETE endpoint | `@app.delete('/delete/{patient_id}')` | `del data[patient_id]` then `save_data` |
| Not found | `raise HTTPException(status_code=404, ...)` | for PUT and DELETE |
| Test client | `client.put(url, json=...)`, `client.delete(url)` | no server needed |

## 🔑 Key ideas
1. **PUT = update, DELETE = remove.** With POST and GET this makes a full **CRUD** API.
2. Use a **separate all-optional model** for partial updates, and `exclude_unset` so unsent fields are not overwritten.
3. After merging, **rebuild the full model**: you get validation and refreshed computed fields for free.
4. **Idempotent** = repeating the request leaves the same final state (GET, PUT, DELETE ✅, POST ❌).
5. Read errors carefully: `404` missing record, `409` duplicate, `422` invalid body, `400` invalid business input.

## ⚠️ Common mistakes
1. Forgetting `save_data(data)` after `update` / `del`: changes vanish.
2. Dumping the update model **without** `exclude_unset`: every unsent field becomes `None`.
3. Putting a default inside `Query(...)`/`Path(...)` **and** after `=` with `Annotated` (crashes the app).
4. Building a Pydantic model manually and not handling `ValidationError` (→ 500).
5. Naming a parameter the same as its function (`update_patient`), which shadows it.
6. Deleting without a `404` check: `del data[missing_id]` raises `KeyError` (→ 500).
7. Trusting the client for `id`, `bmi`, `verdict`: they must not be updatable.
8. Saving data in one format (`'Female'`) and validating with another (`Literal['female']`): normalise input with a `before` validator.

## 🎯 Practice exercises
1. Change `@app.put` to `@app.patch` and explain why that is the more accurate method. Then add a real `PUT` that requires the full record.
2. Make DELETE return `204 No Content` (`status_code=204`, return nothing). How does the test change?
3. Implement a **soft delete**: add `is_deleted: bool = False` and hide deleted patients from `/view`.
4. Add `GET /patients/city/{city}` and `GET /filter?min_age=&max_age=` (optional query params).
5. Add a `model_validator` that rejects `age < 18` combined with `verdict == 'Obese'` and observe the `422`.
6. Replace the JSON file with **SQLite** while keeping the same endpoint signatures.
7. Add authentication (an API-key header) so only authorised clients can `PUT`/`DELETE`.

## 🔗 References
- FastAPI request body & updates: <https://fastapi.tiangolo.com/tutorial/body-updates/>
- FastAPI query parameters with `Annotated`: <https://fastapi.tiangolo.com/tutorial/query-params-str-validations/>
- MDN: PUT <https://developer.mozilla.org/en-US/docs/Web/HTTP/Methods/PUT> · PATCH <https://developer.mozilla.org/en-US/docs/Web/HTTP/Methods/PATCH> · DELETE <https://developer.mozilla.org/en-US/docs/Web/HTTP/Methods/DELETE>